# Practice Lab: Privacy & Data Protection (Lesson 15.4)

Module 15 . 3 exercises . PII scanner + anonymization pipeline + NER enhancement


# Lesson 15.4: Privacy & Data Protection

3 exercises: 1E + 1M + 1C

Module 15: Ethics, Safety, Governance & Explainability


In [ ]:
import re, hashlib
from dataclasses import dataclass
from collections import Counter


---
## Exercise 1 (Easy): PII Scanner


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re
from dataclasses import dataclass
from collections import Counter

@dataclass
class PM:
    t:str; v:str; s:int; e:int

class PD:
    P={"email":r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
       "phone":r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b",
       "aadhaar":r"\b\d{4}[\s-]\d{4}[\s-]\d{4}\b",
       "pan":r"\b[A-Z]{5}\d{4}[A-Z]\b",
       "cc":r"\b\d{4}[\s-]\d{4}[\s-]\d{4}[\s-]\d{4}\b",
       "ip":r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b"}
    def detect(self,text):
        ms=[]
        for t,p in self.P.items():
            for m in re.finditer(p,text): ms.append(PM(t,m.group(),m.start(),m.end()))
        return ms
    def redact(self,text):
        for m in sorted(self.detect(text),key=lambda x:-x.s):
            text=text[:m.s]+f"[{m.t.upper()}]"+text[m.e:]
        return text

d=PD()
texts=["Hi, my email is priya.sharma@gmail.com and phone is 9876543210",
    "Please refund. Contact: rahul@outlook.com, +91-8765432109",
    "My Aadhaar is 1234 5678 9012 and PAN is ABCDE1234F",
    "Card 4532 1234 5678 9012 was charged twice",
    "Server logs from 192.168.1.100 and 10.0.0.55",
    "Send to amit@company.co.in, phone 7654321098",
    "Aadhaar: 9876 5432 1098, PAN: ZYXWV9876A",
    "No personal info in this general inquiry",
    "What is the price of the GenAI bootcamp?",
    "Login from IP 203.0.113.42 at 3AM",
    "Contact: deepa@yahoo.com, Aadhaar 5555 6666 7777",
    "Update phone to 6543210987",
    "PAN card MNOPQ5678R for tax certificate",
    "How do I reset my password?",
    "Course schedule question for next batch",
    "Email: team@netsetos.com and CC: admin@netsetos.com",
    "Credit card 1111 2222 3333 4444 needs removal",
    "Numbers: 8888777766 and backup 9999888877",
    "Aadhaar 1111 2222 3333, PAN LMNOP1234Q, email test@test.com",
    "General question about programming languages"]

print("PII Scanner (20 texts):")
tc=Counter(); tpc=[]
for i,t in enumerate(texts):
    ms=d.detect(t); tpc.append((i+1,len(ms),t[:50]))
    for m in ms: tc[m.t]+=1

print(f"  Total PII: {sum(tc.values())}")
for t,c in tc.most_common(): print(f"    {t:<10} {c:>3} {'#'*c*2}")
clean=sum(1 for _,c,_ in tpc if c==0)
print(f"  Clean: {clean}/{len(texts)} ({clean*100//len(texts)}%)")
print(f"\n  Top-5 redacted:")
for idx,cnt,_ in sorted(tpc,key=lambda x:-x[1])[:5]:
    print(f"    Text {idx}: {d.redact(texts[idx-1])[:65]}...")


</details>


---
## Exercise 2 (Medium): Anonymization Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, hashlib

class AN:
    P={"email":r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
       "phone":r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b",
       "aadhaar":r"\b\d{4}[\s-]\d{4}[\s-]\d{4}\b",
       "pan":r"\b[A-Z]{5}\d{4}[A-Z]\b"}
    def __init__(self): self.fwd={}; self.rev={}
    def _fake(self,t,v):
        k=f"{t}:{v}"
        if k not in self.fwd:
            h=hashlib.md5(v.encode()).hexdigest()[:8]
            f={"email":f"user_{h}@example.com","phone":f"9000{h[:6]}",
               "aadhaar":f"0000 0000 {h[:4]}","pan":f"XXXXX{h[:4].upper()}X"}.get(t,f"[ANON_{h}]")
            self.fwd[k]=f; self.rev[f]=v
        return self.fwd[k]
    def anonymize(self,text):
        for t,p in self.P.items():
            for m in re.finditer(p,text): text=text.replace(m.group(),self._fake(t,m.group()))
        return text
    def deanonymize(self,text):
        for f,r in sorted(self.rev.items(),key=lambda x:-len(x[0])): text=text.replace(f,r)
        return text

a=AN()
tickets=["priya.sharma@gmail.com (9876543210). Aadhaar 1234 5678 9012.",
    "Refund for rahul@outlook.com, PAN ABCDE1234F, +91-8765432109.",
    "Receipt to amit@company.co.in, phone 7654321098.",
    "KYC: Aadhaar 9876 5432 1098, PAN ZYXWV9876A, deepa@yahoo.com.",
    "Contact team@netsetos.com, phone 6543210987.",
    "Aadhaar 5555 6666 7777, email test@test.com.",
    "Update phone 8888777766 for user@domain.com.",
    "PAN MNOPQ5678R, Aadhaar 1111 2222 3333.",
    "Email student@college.edu, phone 9999888877, PAN LMNOP1234Q.",
    "Contact priya.sharma@gmail.com again."]

print("Anonymization Pipeline:")
ok=0
for i,t in enumerate(tickets):
    anon=a.anonymize(t); restored=a.deanonymize(a.anonymize(t))
    match=restored==t
    if match: ok+=1
    if i<3 or i==9:
        print(f"  {i+1}. Original:   {t[:60]}...")
        print(f"     Anonymized: {anon[:60]}...")
        print(f"     Restored:   {'OK' if match else 'FAIL'}")
    elif i==3: print(f"  ... (4-9 processed) ...")

ck=a.fwd.get("email:priya.sharma@gmail.com")
print(f"\n  Consistency: priya.sharma@gmail.com -> {ck}")
print(f"  Restored: {ok}/{len(tickets)} | Mappings: {len(a.fwd)}")


</details>


---
## Exercise 3 (Challenge): NER Enhancement


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re
from collections import Counter

class NER:
    STOP={"the","and","for","from","with","about","this","that","will","have",
        "has","been","was","are","please","hello","thank","dear","regards",
        "subject","order","refund","course","batch","payment","account",
        "issue","problem","request","update","contact","email","phone",
        "send","need","help","regarding","query","kindly","sir","madam",
        "good","morning","afternoon","hi","hey","my","your","no","yes",
        "new","old","general","just","asking","called","needs","verified",
        "inquiry","prospective","student","cancelled","courses","backup"}
    ADDR={"road","rd","street","nagar","colony","lane","avenue","sector",
        "block","flat","marg","layout","enclave"}
    CITIES={"mumbai","delhi","bangalore","bengaluru","hyderabad","chennai",
        "kolkata","pune","noida","gurgaon"}
    def names(self,text):
        ws=text.split(); ns=[]; i=0
        while i<len(ws)-1:
            w1=ws[i].strip(".,!?;:()"); w2=ws[i+1].strip(".,!?;:()")
            if(w1 and w2 and w1[0].isupper() and w2[0].isupper()
               and w1.lower() not in self.STOP and w2.lower() not in self.STOP
               and len(w1)>1 and len(w2)>1):
                ns.append({"type":"person_name","value":f"{w1} {w2}"}); i+=2
            else: i+=1
        return ns
    def addrs(self,text):
        found=[]
        for kw in self.ADDR:
            for m in re.finditer(rf"\b\d+[\s,]*\w*\s*{kw}\b",text,re.I):
                found.append({"type":"address","value":m.group()})
        for c in self.CITIES:
            for m in re.finditer(rf"\b{c}\b",text,re.I):
                found.append({"type":"city","value":m.group()})
        return found

class CD:
    RP={"email":r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "phone":r"\b(?:\+91[\s-]?)?[6-9]\d{9}\b",
        "aadhaar":r"\b\d{4}[\s-]\d{4}[\s-]\d{4}\b",
        "pan":r"\b[A-Z]{5}\d{4}[A-Z]\b"}
    def __init__(self): self.ner=NER()
    def regex_only(self,t):
        return [{"type":pt,"value":m.group()} for pt,p in self.RP.items() for m in re.finditer(p,t)]
    def combined(self,t):
        return self.regex_only(t)+self.ner.names(t)+self.ner.addrs(t)

td=[{"t":"Contact Priya Sharma at priya@gmail.com, phone 9876543210","p":["person_name","email","phone"]},
    {"t":"Rahul Verma lives at 42 MG Road, Bangalore","p":["person_name","address","city"]},
    {"t":"Aadhaar: 1234 5678 9012, PAN: ABCDE1234F for Amit Kumar","p":["aadhaar","pan","person_name"]},
    {"t":"Send to 15 Nehru Nagar, Pune. Contact Deepa Joshi.","p":["address","city","person_name"]},
    {"t":"No PII here. Just asking about course pricing.","p":[]},
    {"t":"Vikram Singh called from 8765432109","p":["person_name","phone"]},
    {"t":"Flat 302, Sector 15, Noida for Meera Patel","p":["address","address","city","person_name"]},
    {"t":"Email: student@college.edu and ravi.k@yahoo.com","p":["email","email"]},
    {"t":"Anjali Deshmukh, 7 Lake Colony, Hyderabad, PAN MNOPQ5678R","p":["person_name","address","city","pan"]},
    {"t":"What is the refund policy?","p":[]},
    {"t":"Sanjay Gupta from Mumbai needs Aadhaar 5555 6666 7777","p":["person_name","city","aadhaar"]},
    {"t":"Address: 88 Brigade Road, Bangalore. Phone: 6543210987","p":["address","city","phone"]},
    {"t":"Kavita Rao at kavita@netsetos.com, 23 Gandhi Marg, Delhi","p":["person_name","email","address","city"]},
    {"t":"General inquiry from a prospective student","p":[]},
    {"t":"Arjun Reddy, Aadhaar 9999 8888 7777, 10 Jubilee Lane, Hyderabad","p":["person_name","aadhaar","address","city"]}]

d=CD()
rtp=rfp=rfn=ctp=cfp=cfn=0
for item in td:
    tt=Counter(item["p"]); rf=Counter(m["type"] for m in d.regex_only(item["t"]))
    cf=Counter(m["type"] for m in d.combined(item["t"]))
    for t,c in tt.items():
        f=rf.get(t,0); rtp+=min(f,c); rfn+=max(0,c-f)
        f2=cf.get(t,0); ctp+=min(f2,c); cfn+=max(0,c-f2)
    for t,c in rf.items():
        if t not in tt: rfp+=c
    for t,c in cf.items():
        if t not in tt: cfp+=c

def pr(tp,fp,fn):
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1); f=2*p*r/max(p+r,0.001)
    return round(p,3),round(r,3),round(f,3)

rp,rr,rf1=pr(rtp,rfp,rfn); cp,cr,cf1=pr(ctp,cfp,cfn)
print("Regex vs Regex+NER:")
print(f"  {'Method':<15} {'Prec':>7} {'Recall':>7} {'F1':>7}")
print(f"  {'Regex':<15} {rp:>7.3f} {rr:>7.3f} {rf1:>7.3f}")
print(f"  {'Regex+NER':<15} {cp:>7.3f} {cr:>7.3f} {cf1:>7.3f}")
print(f"\n  NER adds: person names, addresses, cities")


</details>
